# Prompt 28: Stateful Streaming — Windows and Watermarking
## Databricks Certified Associate Developer for Apache Spark
### Topic 5 — Structured Streaming (10%)

---

## Stateless vs Stateful Operations

| Type | Definition | Examples |
|------|------------|----------|
| **Stateless** | Each row processed independently — no memory of previous rows or micro-batches | `filter()`, `select()`, `map()`, `flatMap()`, `withColumn()` |
| **Stateful** | Processing depends on accumulated state across multiple rows or micro-batches | `groupBy().agg()`, windowed aggregations, `dropDuplicates()`, stream-stream joins |

**Why stateful operations need special handling in streaming:**
- State must be maintained across micro-batches in a fault-tolerant way (stored in state store)
- Without watermarking, state grows unboundedly — eventually causes OOM
- With watermarking, Spark knows when it is safe to evict old state

---

## Event Time vs Processing Time

| | Event Time | Processing Time |
|-|------------|------------------|
| **Definition** | When the event actually occurred (in the data) | When Spark processes the event |
| **Source** | A timestamp column in the data (e.g., `event_ts`) | `current_timestamp()` / system clock |
| **Late data handling** | Handled correctly via watermarking | Late = out of order — impossible to detect |
| **Recommended for** | Production windowed aggregations | Simple stateless operations only |

**Always prefer event time for windowed aggregations** — it correctly handles out-of-order and late-arriving data.

---

## Tumbling Windows

Non-overlapping, fixed-size time windows. Each event belongs to exactly **one** window.

```
Event time:  ─────────────────────────────────────────────────────►
             [  00:00 – 00:05  ][  00:05 – 00:10  ][  00:10 – 00:15  ]
              Tumbling window 1   Tumbling window 2   Tumbling window 3
```

```python
from pyspark.sql.functions import window, col

# Count events per 5-minute tumbling window
df_windowed = df_events \
    .withWatermark('event_ts', '10 minutes') \
    .groupBy(
        window(col('event_ts'), '5 minutes')  # tumbling: only duration
    ) \
    .count()

# Result schema:
# window: struct<start: timestamp, end: timestamp>
# count: long

# Access window start/end
df_windowed.select(
    col('window.start'),
    col('window.end'),
    col('count')
)
```

**Signature:** `window(timeColumn, windowDuration)` — window duration as a string (e.g., `'5 minutes'`, `'1 hour'`, `'30 seconds'`)

---

## Sliding Windows

Overlapping windows. Each event may belong to **multiple** windows. Defined by both a duration and a slide interval.

```
Event time:  ──────────────────────────────────────────────────────►
             [──── 10 min ────]
                  [──── 10 min ────]
                       [──── 10 min ────]
             ◄──────► ◄──────► ◄──────►
              5 min    5 min    5 min
              slide    slide    slide
```

```python
# 10-minute window, sliding every 5 minutes
# An event at 00:07 belongs to BOTH the 00:00-00:10 AND 00:05-00:15 windows
df_sliding = df_events \
    .withWatermark('event_ts', '10 minutes') \
    .groupBy(
        window(col('event_ts'), '10 minutes', '5 minutes')  # duration, slide
    ) \
    .agg(count('event_id').alias('event_count'), avg('amount').alias('avg_amount'))
```

**Signature:** `window(timeColumn, windowDuration, slideDuration)` — slide must be ≤ window duration

**Rule:** If slide < duration → overlapping (sliding). If slide == duration → non-overlapping (tumbling). No `slideDuration` argument → tumbling.

---

## Session Windows (Spark 3.2+)

Dynamic-size windows based on activity gaps. A session window extends as long as events keep arriving within a gap duration. If no event arrives within the gap, the session closes.

```
Events:  e1  e2  e3         e4  e5           e6
          ├──┼──┤ gap>5min   ├──┤ gap>5min    ├─ new session
          Session 1          Session 2         Session 3
```

```python
from pyspark.sql.functions import session_window

df_sessions = df_events \
    .withWatermark('event_ts', '10 minutes') \
    .groupBy(
        session_window(col('event_ts'), '5 minutes'),  # gap duration
        col('user_id')
    ) \
    .agg(count('event_id').alias('actions'), _sum('amount').alias('session_revenue'))
```

---

## Watermarking: withWatermark()

### What is a watermark?

A watermark tells Spark: **"any data arriving more than X time late (relative to the maximum event time seen so far) should be considered late and may be dropped."**

```
Max event time seen so far:  12:00
Watermark delay:              10 minutes
Current watermark threshold:  11:50

→ Events with event_ts < 11:50 are considered late and may be dropped
→ State for windows ending before 11:50 can be safely evicted
```

### Syntax

```python
df_with_watermark = df_stream.withWatermark('event_ts', '10 minutes')
# Must be called BEFORE groupBy() for the watermark to apply to the aggregation
```

### Why watermarks matter (exam critical)

| Without watermark | With watermark |
|-------------------|----------------|
| State grows unboundedly (every window kept forever) | State evicted after window + delay period |
| Only `complete` and `update` output modes valid | All three modes valid (`append` allowed) |
| No late-data handling — all data included | Late data beyond delay is dropped |

### Watermark + output mode compatibility (exam critical)

| Output Mode | Without watermark | With watermark |
|-------------|-------------------|----------------|
| `append` | ❌ | ✅ (rows emitted only when window finalised) |
| `complete` | ✅ | ✅ |
| `update` | ✅ | ✅ |

**Key insight for `append` + watermark:** A windowed row is emitted to the output **only when the watermark advances past the window's end time** — guaranteeing the window is final and no more late data can arrive for it.

### How the watermark threshold advances

```
Trigger 1: max event_ts seen = 12:05  → watermark = 12:05 - 10min = 11:55
Trigger 2: max event_ts seen = 12:20  → watermark = 12:20 - 10min = 12:10
Trigger 3: max event_ts seen = 12:18  → watermark STAYS at 12:10 (never goes backward)
```

The watermark is **monotonically non-decreasing** — it only moves forward.

---

## Stateful Deduplication with Watermarks

```python
# Deduplicate events within a time window — drop events with duplicate event_id
# within a 1-hour window (events arriving more than 1h late are not considered duplicates)
df_deduped = df_stream \
    .withWatermark('event_ts', '1 hour') \
    .dropDuplicates(['event_id', 'event_ts'])

# Without watermark: state kept forever — would eventually OOM
# With watermark: state for events older than (max_event_ts - 1h) is evicted
```

**Supported output modes for `dropDuplicates` with watermark:** `append` only.

---

## Window Duration String Format

Valid duration strings for `window()` and `withWatermark()`:

```
'1 second'    '30 seconds'
'1 minute'    '5 minutes'    '30 minutes'
'1 hour'      '6 hours'      '12 hours'
'1 day'       '7 days'
```

Also supports compound: `'1 hour 30 minutes'`

---

## Accessing Window Start and End

The `window` column in the result is a `struct<start: timestamp, end: timestamp>`:

```python
df_result.select(
    col('window.start').alias('window_start'),
    col('window.end').alias('window_end'),
    col('count')
).show()
```

In [ ]:
# Cell 1: Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, window, count, sum as _sum, avg, expr, lit,
    to_timestamp, current_timestamp, explode, split
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType,
    DoubleType, TimestampType, LongType
)
import os, shutil, time, json

spark = SparkSession.builder \
    .appName('StatefulStreamingWindows') \
    .master('local[2]') \
    .config('spark.sql.shuffle.partitions', '2') \
    .getOrCreate()

spark.sparkContext.setLogLevel('ERROR')

def clean_dir(path):
    if os.path.exists(path):
        shutil.rmtree(path)
    os.makedirs(path)

print('Setup complete')

In [ ]:
# Cell 2: Batch simulation of window functions (same API, results verifiable)
# Structured Streaming window() uses the SAME function as batch.
# We demonstrate on a batch DataFrame first so results are immediately visible.

from datetime import datetime, timedelta

# Create events with explicit event timestamps spanning ~30 minutes
base = datetime(2026, 4, 22, 12, 0, 0)
events = [
    # (event_id, user_id, event_type, amount, event_ts)
    (1,  1, 'click',    0.0,  base + timedelta(minutes=2)),
    (2,  2, 'purchase', 29.99, base + timedelta(minutes=3)),
    (3,  1, 'click',    0.0,  base + timedelta(minutes=4)),
    (4,  3, 'purchase', 59.99, base + timedelta(minutes=7)),
    (5,  2, 'click',    0.0,  base + timedelta(minutes=8)),
    (6,  4, 'purchase', 14.99, base + timedelta(minutes=11)),
    (7,  1, 'click',    0.0,  base + timedelta(minutes=12)),
    (8,  3, 'view',     0.0,  base + timedelta(minutes=13)),
    (9,  5, 'purchase', 99.99, base + timedelta(minutes=17)),
    (10, 2, 'click',    0.0,  base + timedelta(minutes=22)),
    (11, 4, 'purchase', 45.00, base + timedelta(minutes=24)),
    (12, 1, 'view',     0.0,  base + timedelta(minutes=28)),
]

schema = StructType([
    StructField('event_id',   IntegerType(), False),
    StructField('user_id',    IntegerType(), True),
    StructField('event_type', StringType(),  True),
    StructField('amount',     DoubleType(),  True),
    StructField('event_ts',   TimestampType(), True),
])

df_events = spark.createDataFrame(events, schema)
print('=== Raw events ===')
df_events.show(truncate=False)

# -------------------------------------------------------
# 1. Tumbling window: 10-minute non-overlapping windows
# -------------------------------------------------------
print('=== Tumbling window: 10 minutes ===')
df_tumbling = df_events \
    .groupBy(window(col('event_ts'), '10 minutes')) \
    .agg(
        count('event_id').alias('event_count'),
        _sum('amount').alias('total_revenue')
    ) \
    .select(
        col('window.start').alias('window_start'),
        col('window.end').alias('window_end'),
        col('event_count'),
        col('total_revenue')
    ) \
    .orderBy('window_start')

df_tumbling.show(truncate=False)
# 12:00-12:10 → events 1-5
# 12:10-12:20 → events 6-9
# 12:20-12:30 → events 10-12

# -------------------------------------------------------
# 2. Sliding window: 10-minute windows sliding every 5 minutes
# -------------------------------------------------------
print('=== Sliding window: 10 min window, 5 min slide ===')
df_sliding = df_events \
    .groupBy(window(col('event_ts'), '10 minutes', '5 minutes')) \
    .agg(
        count('event_id').alias('event_count'),
        _sum('amount').alias('total_revenue')
    ) \
    .select(
        col('window.start').alias('window_start'),
        col('window.end').alias('window_end'),
        col('event_count'),
        col('total_revenue')
    ) \
    .orderBy('window_start')

df_sliding.show(truncate=False)
print('Note: event at 12:07 appears in BOTH 12:00-12:10 AND 12:05-12:15 windows (overlapping)')

In [ ]:
# Cell 3: Watermarking in streaming context using rate + file sources
# Demonstrates withWatermark() + window aggregation + output mode behaviour

STREAM_INPUT = '/tmp/spark_watermark_input'
STREAM_CKPT  = '/tmp/spark_watermark_ckpt'
clean_dir(STREAM_INPUT)
clean_dir(STREAM_CKPT)

event_schema = StructType([
    StructField('event_id',   IntegerType(), True),
    StructField('event_type', StringType(),  True),
    StructField('amount',     DoubleType(),  True),
    StructField('event_ts',   StringType(),  True),  # string → cast to timestamp
])

# -------------------------------------------------------
# Streaming query with watermark + tumbling window
# Output mode: complete (write full result every trigger)
# -------------------------------------------------------
df_stream = spark.readStream \
    .format('json') \
    .schema(event_schema) \
    .option('maxFilesPerTrigger', 1) \
    .load(STREAM_INPUT)

df_parsed = df_stream.withColumn('event_ts', to_timestamp(col('event_ts'), 'yyyy-MM-dd HH:mm:ss'))

df_windowed = df_parsed \
    .withWatermark('event_ts', '5 minutes') \
    .groupBy(
        window(col('event_ts'), '10 minutes')
    ) \
    .agg(
        count('event_id').alias('event_count'),
        _sum('amount').alias('total_revenue')
    )

query = df_windowed.writeStream \
    .format('memory') \
    .queryName('windowed_stream') \
    .outputMode('complete') \
    .option('checkpointLocation', STREAM_CKPT) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(3)

# Batch 1: events in 12:00-12:10 window
batch1 = [
    {'event_id': 1, 'event_type': 'click',    'amount': 0.0,  'event_ts': '2026-04-22 12:02:00'},
    {'event_id': 2, 'event_type': 'purchase', 'amount': 29.99,'event_ts': '2026-04-22 12:04:00'},
    {'event_id': 3, 'event_type': 'click',    'amount': 0.0,  'event_ts': '2026-04-22 12:07:00'},
]
with open(os.path.join(STREAM_INPUT, 'batch1.json'), 'w') as f:
    for r in batch1: f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== After batch 1 (events at 12:02-12:07, window 12:00-12:10) ===')
spark.sql('SELECT window.start, window.end, event_count, total_revenue FROM windowed_stream ORDER BY window.start').show(truncate=False)

# Batch 2: events in 12:10-12:20 window
batch2 = [
    {'event_id': 4, 'event_type': 'purchase', 'amount': 59.99,'event_ts': '2026-04-22 12:12:00'},
    {'event_id': 5, 'event_type': 'click',    'amount': 0.0,  'event_ts': '2026-04-22 12:15:00'},
]
with open(os.path.join(STREAM_INPUT, 'batch2.json'), 'w') as f:
    for r in batch2: f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== After batch 2 (events at 12:12-12:15, window 12:10-12:20) ===')
spark.sql('SELECT window.start, window.end, event_count, total_revenue FROM windowed_stream ORDER BY window.start').show(truncate=False)

# Batch 3: late event for the 12:00-12:10 window — arrives at processing time but
# event_ts is 12:08, and max event_ts so far is 12:15, watermark = 12:10
# 12:08 > 12:10 is FALSE → this event is within watermark, will be included
batch3 = [
    {'event_id': 6, 'event_type': 'view', 'amount': 0.0, 'event_ts': '2026-04-22 12:08:00'},
]
with open(os.path.join(STREAM_INPUT, 'batch3.json'), 'w') as f:
    for r in batch3: f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== After batch 3 (late event at 12:08 — within 5-min watermark, should be counted) ===')
spark.sql('SELECT window.start, window.end, event_count, total_revenue FROM windowed_stream ORDER BY window.start').show(truncate=False)
print('Note: 12:00-12:10 window now shows 4 events (late event at 12:08 was accepted)')

query.stop()
print('\nStream stopped')

In [ ]:
# Cell 4: Stateful deduplication with watermark

DEDUP_INPUT = '/tmp/spark_dedup_input'
DEDUP_CKPT  = '/tmp/spark_dedup_ckpt'
clean_dir(DEDUP_INPUT)
clean_dir(DEDUP_CKPT)

dedup_schema = StructType([
    StructField('event_id',   StringType(),  True),
    StructField('user_id',    IntegerType(), True),
    StructField('event_type', StringType(),  True),
    StructField('event_ts',   StringType(),  True),
])

df_raw = spark.readStream \
    .format('json') \
    .schema(dedup_schema) \
    .load(DEDUP_INPUT)

df_deduped = df_raw \
    .withColumn('event_ts', to_timestamp(col('event_ts'), 'yyyy-MM-dd HH:mm:ss')) \
    .withWatermark('event_ts', '10 minutes') \
    .dropDuplicates(['event_id', 'event_ts'])  # deduplicate by event_id within watermark

# output mode must be append for dropDuplicates with watermark
q_dedup = df_deduped.writeStream \
    .format('memory') \
    .queryName('dedup_stream') \
    .outputMode('append') \
    .option('checkpointLocation', DEDUP_CKPT) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(3)

# First batch: 3 events including one duplicate (event_id = 'evt-001' appears twice)
batch1 = [
    {'event_id': 'evt-001', 'user_id': 1, 'event_type': 'click',    'event_ts': '2026-04-22 13:00:00'},
    {'event_id': 'evt-002', 'user_id': 2, 'event_type': 'purchase', 'event_ts': '2026-04-22 13:01:00'},
    {'event_id': 'evt-001', 'user_id': 1, 'event_type': 'click',    'event_ts': '2026-04-22 13:00:00'},  # duplicate
]
with open(os.path.join(DEDUP_INPUT, 'batch1.json'), 'w') as f:
    for r in batch1: f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== After batch 1 (3 rows, 1 duplicate) ===')
spark.sql('SELECT * FROM dedup_stream ORDER BY event_ts').show()
count1 = spark.sql('SELECT COUNT(*) as n FROM dedup_stream').collect()[0]['n']
print(f'Row count: {count1} (expected 2 — duplicate dropped)')

# Second batch: new events + a duplicate of evt-001 (same event_id, same ts)
batch2 = [
    {'event_id': 'evt-003', 'user_id': 3, 'event_type': 'view',     'event_ts': '2026-04-22 13:05:00'},
    {'event_id': 'evt-001', 'user_id': 1, 'event_type': 'click',    'event_ts': '2026-04-22 13:00:00'},  # duplicate again
    {'event_id': 'evt-004', 'user_id': 4, 'event_type': 'purchase', 'event_ts': '2026-04-22 13:06:00'},
]
with open(os.path.join(DEDUP_INPUT, 'batch2.json'), 'w') as f:
    for r in batch2: f.write(json.dumps(r) + '\n')

time.sleep(4)
print('\n=== After batch 2 (3 rows, 1 duplicate of evt-001) ===')
spark.sql('SELECT * FROM dedup_stream ORDER BY event_ts').show()
count2 = spark.sql('SELECT COUNT(*) as n FROM dedup_stream').collect()[0]['n']
print(f'Row count: {count2} (expected 4 — second duplicate of evt-001 dropped)')

q_dedup.stop()
print('\nDeduplication stream stopped')

In [ ]:
# Cell 5: Output mode impact on when windowed results are emitted
# Demonstrates append vs complete for windowed aggregations with watermark

APPEND_INPUT = '/tmp/spark_append_window_input'
APPEND_CKPT  = '/tmp/spark_append_window_ckpt'
clean_dir(APPEND_INPUT)
clean_dir(APPEND_CKPT)

win_schema = StructType([
    StructField('event_id', IntegerType(), True),
    StructField('event_ts', StringType(),  True),
    StructField('value',    DoubleType(),  True),
])

df_ws = spark.readStream \
    .format('json') \
    .schema(win_schema) \
    .load(APPEND_INPUT)

df_win_agg = df_ws \
    .withColumn('event_ts', to_timestamp(col('event_ts'), 'yyyy-MM-dd HH:mm:ss')) \
    .withWatermark('event_ts', '5 minutes') \
    .groupBy(window(col('event_ts'), '10 minutes')) \
    .agg(count('event_id').alias('n'), _sum('value').alias('total'))

# Use append mode — rows emitted ONLY when window is finalised (watermark past window.end)
q_append = df_win_agg.writeStream \
    .format('memory') \
    .queryName('append_windows') \
    .outputMode('append') \
    .option('checkpointLocation', APPEND_CKPT) \
    .trigger(processingTime='2 seconds') \
    .start()

time.sleep(3)

# Phase 1: events in 10:00-10:10 window
with open(os.path.join(APPEND_INPUT, 'p1.json'), 'w') as f:
    for r in [
        {'event_id': 1, 'event_ts': '2026-04-22 10:02:00', 'value': 10.0},
        {'event_id': 2, 'event_ts': '2026-04-22 10:05:00', 'value': 20.0},
    ]:
        f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== Phase 1: events in 10:00-10:10 window, max_ts=10:05, watermark=10:00 ===')
print('Append mode: 10:00-10:10 window NOT yet emitted (watermark has not passed window.end=10:10)')
spark.sql('SELECT window.start, window.end, n, total FROM append_windows').show()
# Expect: 0 rows — window not yet finalised

# Phase 2: events in 10:20-10:30 window — advances watermark past 10:10
with open(os.path.join(APPEND_INPUT, 'p2.json'), 'w') as f:
    for r in [
        {'event_id': 3, 'event_ts': '2026-04-22 10:22:00', 'value': 30.0},
        {'event_id': 4, 'event_ts': '2026-04-22 10:25:00', 'value': 40.0},
    ]:
        f.write(json.dumps(r) + '\n')

time.sleep(4)
print('=== Phase 2: events in 10:20-10:30 window, max_ts=10:25, watermark=10:20 ===')
print('Watermark=10:20 is now past window.end=10:10 → 10:00-10:10 window is now EMITTED in append mode')
spark.sql('SELECT window.start, window.end, n, total FROM append_windows ORDER BY window.start').show(truncate=False)
# Expect: 10:00-10:10 row with n=2, total=30.0
# 10:20-10:30 not yet emitted (watermark hasn't passed its end)

q_append.stop()

# Summary
print('\n=== Summary: When are windowed rows emitted? ===')
print('append mode:   Only when watermark advances PAST the window end time')
print('               → rows appear with a delay but are GUARANTEED final')
print('complete mode: Every trigger — the entire result table including in-progress windows')
print('               → rows appear immediately but may still be updated')
print('update mode:   Every trigger — only rows that changed (in-progress windows are emitted each trigger)')

spark.stop()
print('\nDone.')

## Real-World Scenarios

<details>
<summary>Scenario 1: Counting page views per 5-minute tumbling window from a Kafka stream</summary>

**Situation:**
A web analytics pipeline reads page view events from Kafka and needs to report the count per page per 5-minute interval for a real-time dashboard.

**Solution:**
```python
from pyspark.sql.functions import from_json, window, col, count

event_schema = StructType([
    StructField('user_id',    StringType(),    True),
    StructField('page',       StringType(),    True),
    StructField('event_time', TimestampType(), True),
])

df_kafka = spark.readStream \
    .format('kafka') \
    .option('kafka.bootstrap.servers', 'broker:9092') \
    .option('subscribe', 'page-views') \
    .load() \
    .selectExpr('CAST(value AS STRING) as json_str') \
    .select(from_json(col('json_str'), event_schema).alias('d')) \
    .select('d.*')

df_page_counts = df_kafka \
    .withWatermark('event_time', '2 minutes') \
    .groupBy(
        window(col('event_time'), '5 minutes'),
        col('page')
    ) \
    .count()

query = df_page_counts.writeStream \
    .outputMode('update') \
    .format('console') \
    .trigger(processingTime='30 seconds') \
    .option('checkpointLocation', '/ckpt/page-counts') \
    .start()
```

**Why `update` mode:** The dashboard is updated as new page views arrive — we want to see partial counts for the current window, and only emit rows that have changed. `complete` would rewrite all historical windows every 30 seconds.

**Exam Sub-topic:** tumbling window syntax; `withWatermark` before `groupBy`; `update` mode for live dashboards
</details>

<details>
<summary>Scenario 2: Detecting a sales spike using a sliding window (rolling average)</summary>

**Situation:**
A fraud detection system needs to compute the total transaction amount in a rolling 1-hour window, updated every 15 minutes, to flag spikes above a threshold.

**Solution:**
```python
df_transactions = (spark.readStream
    .format('kafka') ...)

df_rolling = df_transactions \
    .withWatermark('txn_ts', '15 minutes') \
    .groupBy(
        window(col('txn_ts'), '1 hour', '15 minutes'),  # 1h window, 15min slide
        col('merchant_id')
    ) \
    .agg(_sum('amount').alias('rolling_total'))

# Each event appears in 4 windows (1h / 15min = 4) — correct for a rolling calculation

def flag_spikes(batch_df, batch_id):
    spikes = batch_df.filter(col('rolling_total') > 100_000)
    if spikes.count() > 0:
        spikes.show()
        # send alert...

df_rolling.writeStream \
    .foreachBatch(flag_spikes) \
    .outputMode('update') \
    .option('checkpointLocation', '/ckpt/fraud-rolling') \
    .trigger(processingTime='5 minutes') \
    .start()
```

**Key insight:** A sliding window with `windowDuration='1 hour'` and `slideDuration='15 minutes'` means each event belongs to up to 4 overlapping windows — it is counted in the rolling total for all 4.

**Exam Sub-topic:** sliding window 3-argument form; overlap count = `windowDuration / slideDuration`; `foreachBatch` for custom logic
</details>

<details>
<summary>Scenario 3: Understanding why a windowed query fails with append mode when watermark is missing</summary>

**Situation:**
A developer writes:
```python
df.groupBy(window('event_ts', '10 minutes')).count() \
  .writeStream.outputMode('append').start()
```
The query fails: `AnalysisException: Append output mode not supported when there are streaming aggregations on streaming DataFrames/Datasets without watermark.`

**Explanation:**
Without a watermark, Spark cannot determine when a window is "closed" (i.e., when no more late data can arrive for it). Since the window might receive more data in any future micro-batch, Spark cannot safely emit the row as a final append — it could need to be updated.

**Fix:**
```python
df.withWatermark('event_ts', '10 minutes') \
  .groupBy(window('event_ts', '10 minutes')) \
  .count() \
  .writeStream.outputMode('append').start()
# Now: rows are emitted only when watermark advances past window.end
# Guarantee: the window is final — no more late data will arrive for it
```

**Exam Sub-topic:** `append` mode requires watermark for aggregations; watermark provides finalisation guarantee; `withWatermark` must come BEFORE `groupBy`
</details>

<details>
<summary>Scenario 4: Streaming deduplication to prevent duplicate event processing</summary>

**Situation:**
A data pipeline reads from Kafka, where the same event may be delivered multiple times due to producer retries (at-least-once delivery). The downstream system requires exactly-once semantics.

**Solution:**
```python
df_raw = (spark.readStream.format('kafka')
    .option('kafka.bootstrap.servers', 'broker:9092')
    .option('subscribe', 'orders')
    .load()
    .selectExpr('CAST(value AS STRING) as json_str')
    .select(from_json(col('json_str'), order_schema).alias('d'))
    .select('d.*'))

# Deduplicate by order_id within a 2-hour watermark
# Assumption: duplicate events arrive within 2 hours of the original
df_deduped = df_raw \
    .withWatermark('order_ts', '2 hours') \
    .dropDuplicates(['order_id', 'order_ts'])

df_deduped.writeStream \
    .format('delta') \
    .outputMode('append') \
    .option('checkpointLocation', '/ckpt/orders-dedup') \
    .start('/data/orders-deduped')
```

**Key point:** The watermark controls the deduplication window — duplicates arriving within 2 hours are detected. After the watermark advances past an `order_ts`, the state for that event is evicted and a duplicate arriving after that would be treated as a new event.

**Exam Sub-topic:** `dropDuplicates()` for streaming dedup; watermark required for state eviction; `append` output mode; state store bounded by watermark
</details>

<details>
<summary>Scenario 5: Late data handling — comparing what gets included vs dropped by watermark</summary>

**Situation:**
A developer wants to understand exactly which late-arriving events are included vs dropped by a 10-minute watermark.

**Timeline analysis:**
```
Watermark delay: 10 minutes
Window: 5-minute tumbling

Trigger 1:
  Events processed: event_ts = 14:00, 14:03
  Max event_ts = 14:03
  Watermark threshold = 14:03 - 10min = 13:53
  → events with event_ts >= 13:53 are accepted

Trigger 2:
  Events processed: event_ts = 14:08
  Max event_ts = 14:08
  Watermark threshold = 14:08 - 10min = 13:58
  → Late event at 14:01 arrives:  14:01 >= 13:58 → ACCEPTED ✅
  → Late event at 13:57 arrives:  13:57 < 13:58  → DROPPED  ❌

Trigger 3:
  Events processed: event_ts = 14:20
  Watermark threshold = 14:10
  → Window 14:00-14:05 ends at 14:05 < 14:10 → window is NOW FINAL
  → In append mode: window 14:00-14:05 result is NOW EMITTED
```

**Code to verify:**
```python
df_stream.withWatermark('event_ts', '10 minutes') \
         .groupBy(window(col('event_ts'), '5 minutes')) \
         .count() \
         .writeStream \
         .outputMode('append')  # emits only when window is past watermark
```

**Exam Sub-topic:** watermark threshold = `max_event_ts - delay`; late event acceptance check: `event_ts >= threshold`; append mode emission: when `window.end <= watermark`; watermark is monotonically non-decreasing
</details>

## Exam Cheat Sheet

| Question | Answer |
|----------|--------|
| Stateless operation examples | `filter()`, `select()`, `withColumn()`, `map()` |
| Stateful operation examples | `groupBy().agg()`, windowed aggregations, `dropDuplicates()`, stream-stream joins |
| Tumbling window syntax | `window(col('ts'), '10 minutes')` — 2-arg form |
| Sliding window syntax | `window(col('ts'), '10 minutes', '5 minutes')` — 3-arg form |
| Session window syntax | `session_window(col('ts'), '5 minutes')` — Spark 3.2+ |
| Event belongs to how many tumbling windows? | Exactly **1** |
| Event belongs to how many sliding windows? | Up to `windowDuration / slideDuration` |
| Window result column type | `struct<start: timestamp, end: timestamp>` |
| Access window start | `col('window.start')` |
| Watermark syntax | `df.withWatermark('ts_col', '10 minutes')` — must be BEFORE `groupBy` |
| Watermark threshold formula | `max_event_ts_seen - delay` |
| Watermark direction | Monotonically non-decreasing (never goes backward) |
| Late event accepted? | Yes if `event_ts >= watermark_threshold` |
| Late event dropped? | Yes if `event_ts < watermark_threshold` |
| Without watermark: output modes for aggregation | `complete` and `update` only |
| With watermark: all output modes valid? | **Yes** — `append`, `complete`, `update` all valid |
| Append + watermark: when is window row emitted? | When `watermark_threshold > window.end` (window is finalised) |
| Without watermark: state behaviour | Unbounded growth — eventually OOM |
| With watermark: state behaviour | Evicted when window end < watermark threshold |
| `dropDuplicates` with watermark output mode | `append` only |
| `dropDuplicates` without watermark | Unbounded state — will OOM in long-running stream |

---

## Practice Q&A

<details>
<summary>Q1: What is the difference between a tumbling window and a sliding window? Give the syntax for each.</summary>

**A:**

| | Tumbling | Sliding |
|-|----------|---------|
| Overlap | None — windows are adjacent, non-overlapping | Yes — windows overlap |
| Event membership | Each event belongs to **exactly 1** window | Each event belongs to up to `windowDuration / slideDuration` windows |
| Use case | Simple period aggregations (hourly sales totals) | Rolling/moving averages |

```python
# Tumbling: 10-minute non-overlapping windows
df.groupBy(window(col('ts'), '10 minutes')).count()

# Sliding: 10-minute windows, updated every 5 minutes
df.groupBy(window(col('ts'), '10 minutes', '5 minutes')).count()
# An event at 12:07 belongs to BOTH 12:00-12:10 AND 12:05-12:15
```
</details>

<details>
<summary>Q2: Why does state grow unboundedly without a watermark in a stateful streaming query?</summary>

**A:** For a stateful operation like a windowed aggregation, Spark must remember the partial results for each open window so it can update them when new data arrives for that window.

Without a watermark, Spark has no way to know when a window is "done" — late data could arrive for any past window at any time. Spark must therefore **keep all window state forever**, just in case late data arrives for any of them. In a long-running stream, the number of windows accumulates indefinitely, eventually exhausting executor memory.

With `withWatermark('ts', '10 minutes')`, Spark knows: "once the watermark advances past a window's end time, no more late data can arrive for that window." The state for that window can be safely evicted, bounding memory usage to roughly `(max lateness + window duration) / window duration` windows at any time.
</details>

<details>
<summary>Q3: When does a windowed row appear in the output with append mode vs complete mode?</summary>

**A:**

**`append` mode (requires watermark):**
A windowed row is emitted **only when the watermark threshold advances past the window's end time**. This guarantees the window is final — no more late data can update it. The row appears with a latency of approximately `windowDuration + watermarkDelay`.

**`complete` mode:**
The **entire result table** (all windows seen so far, including in-progress ones) is emitted **every trigger**. Windows appear in the output immediately on the first trigger where they have data, but their counts may still increase as more data arrives in future triggers.

**`update` mode:**
Only the windows that **changed** in the current trigger are emitted. In-progress windows appear every trigger as their count increases. Like `complete`, rows may be emitted before the window is final.
</details>

<details>
<summary>Q4: Given a 10-minute watermark delay and current max event_ts = 14:25, which of these events are accepted or dropped?  (a) event_ts = 14:20  (b) event_ts = 14:10  (c) event_ts = 14:14</summary>

**A:**
Watermark threshold = max_event_ts - delay = 14:25 - 10min = **14:15**

An event is accepted if `event_ts >= watermark_threshold`:
- (a) 14:20 ≥ 14:15 → **ACCEPTED** ✅
- (b) 14:10 < 14:15 → **DROPPED** ❌
- (c) 14:14 < 14:15 → **DROPPED** ❌

The borderline is 14:15 — events with event_ts exactly equal to the threshold are accepted.
</details>

<details>
<summary>Q5: What output mode is required for streaming deduplication with dropDuplicates() + watermark?</summary>

**A:** **`append`** mode only.

```python
df_deduped = df_stream \
    .withWatermark('event_ts', '1 hour') \
    .dropDuplicates(['event_id', 'event_ts'])

df_deduped.writeStream \
    .outputMode('append')  # required
    ...
```

`complete` and `update` are not supported for `dropDuplicates`. The `append` mode is appropriate because each deduplicated event is emitted once (when it is confirmed unique after the watermark advances) and never needs to be updated or retracted.

**Without watermark:** `dropDuplicates()` works on a stream but state grows unboundedly — each unique `(event_id, event_ts)` pair is remembered forever. This is fine for small key spaces but will OOM for large unbounded streams. Always use a watermark with `dropDuplicates` in production.
</details>